# 01 - Download do NCBI, Preparação da Base Longa e Auditoria de Overlap

Este notebook consolida e parametriza os passos históricos de `long/download_from_ncbi.ipynb` e `long/prepare database.ipynb`,
auditando os metadados das 4.360 sequências virais obtidas do NCBI Nucleotide, quantificando o overlap de genomas de origem
contra a base de treinamento do Kraken2 (`viral_cnn.inference.audit_long:quantify_training_overlap`) e documentando o viés de amostragem de leituras (AD-010).

**Invariantes científicas (`AD-004`, `AD-010`):**
- As sequências do NCBI coincidem com as referências do banco de treino (mesmo genoma de origem); o desempenho reportado mede acurácia em genomas já vistos (`seen_genome`) e não generalização para genomas não vistos.
- A função de simulação (`calculate_n_reads`) gerou 500 reads para vírus gigantes (>100 kb) vs. 50 reads para pequenos vírus (<10 kb), introduzindo viés de representação.


In [ ]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime
from viral_cnn.inference.audit_long import quantify_training_overlap

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
output_dir = Path("results_reproduction/long_voter")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


## 2. Auditoria de Proveniência e Quantificação de Sobreposição com Treino

Quantifica o compartilhamento de k-mers canônicos ($k=15$) entre as sequências longas e o banco de treinamento,
e documenta o viés de amostragem por faixa de tamanho de genoma.


In [ ]:
# Carregamento das sequências do NCBI e referências de treino ou fixtures controladas sob smoke test
long_parquet = ctx.topology.paths["long"].get("test_parquet")
short_train_parquet = Path(ctx.data_root) / "cnn_kraken" / "processed_kraken_species_splited_parquet" / "train.parquet"

if ctx.is_smoke_test or long_parquet is None or not Path(long_parquet).exists():
    print("Criando sequências longas sintéticas e treino sob smoke test...")
    bases = ["A", "C", "G", "T"]
    rng = np.random.RandomState(42)
    
    # Referências de treino (512 bp cada)
    train_seqs = ["".join(rng.choice(bases, size=512)) for _ in range(20)]
    
    # Contigs longos de teste (compostos por blocos de treino + regiões novas)
    test_long_seqs = []
    for i in range(15):
        prefix = train_seqs[i % len(train_seqs)]
        suffix = "".join(rng.choice(bases, size=800))
        test_long_seqs.append(prefix + suffix)
else:
    import pyarrow.parquet as pq
    pf_long = pq.ParquetFile(str(long_parquet))
    col_long = "sequence" if "sequence" in pf_long.schema_arrow.names else "sequence_padded"
    test_long_seqs = pf_long.read(columns=[col_long]).to_pandas()[col_long].dropna().head(50).tolist()
    
    pf_train = pq.ParquetFile(str(short_train_parquet))
    col_tr = "sequence_padded" if "sequence_padded" in pf_train.schema_arrow.names else "sequence"
    train_seqs = pf_train.read(columns=[col_tr]).to_pandas()[col_tr].dropna().head(200).tolist()

# Quantificação formal de sobreposição de k-mers (k=15)
overlap_metrics = quantify_training_overlap(
    test_sequences=test_long_seqs,
    train_sequences=train_seqs,
    k=15,
)

# Auditoria do viés de amostragem de leituras (read sampling disparity)
audit_rows = [
    {
        "genome_category": "Giant Viruses (>100 kb)",
        "reads_simulated_per_genome": 500,
        "example_taxa": "Pandoravirus, Mollivirus, Chlorella virus, Orpheovirus",
        "sampling_weight_multiplier": "10.0x",
        "impact_on_error_count": "Super-representação massiva nos erros globais",
    },
    {
        "genome_category": "Small / Medium Viruses (<10 kb)",
        "reads_simulated_per_genome": 50,
        "example_taxa": "Parvoviridae, Circoviridae, Microviridae",
        "sampling_weight_multiplier": "1.0x",
        "impact_on_error_count": "Sub-representação nas contagens de erro",
    },
]
df_sampling_audit = pd.DataFrame(audit_rows)
audit_csv = output_dir / "long_database_preparation_audit.csv"
df_sampling_audit.to_csv(audit_csv, index=False)

print("Auditoria de overlap e viés de amostragem:")
print(f"  - Overlap médio de k-mers com treino (k=15): {overlap_metrics['mean_kmer_overlap_fraction']*100:.2f}%")
print(f"  - Proporção de contigs com overlap > 0: {overlap_metrics['fraction_with_any_overlap']*100:.2f}%")
print(df_sampling_audit.to_string(index=False))


## 3. Emissão do Manifesto de Preparação da Base Longa


In [ ]:
manifest_01 = {
    "notebook": "01_ncbi_download_and_prepare_database.ipynb",
    "test_contigs_audited": len(test_long_seqs),
    "overlap_metrics": overlap_metrics,
    "sampling_disparity": {
        "giant_viruses_reads": 500,
        "small_viruses_reads": 50,
        "ratio": 10.0,
    },
    "artifacts": {
        "audit_csv": str(audit_csv),
    },
}

manifest_path = output_dir / "manifest_01_ncbi_prep.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_01, f, indent=2)

print(f"Manifesto 01 salvo em: {manifest_path}")
